In [1]:
import os
import numpy as np
from PIL import Image              #NumPy array ko image me convert karke save karne ke liye.

os.makedirs("dataset/real", exist_ok=True)  #Ye 2 folders banata hai.  dataset ->real/fake
os.makedirs("dataset/fake", exist_ok=True)

In [2]:
for i in range(50):
    img = np.random.randint(100, 255, (128,128,3), dtype=np.uint8)   
    Image.fromarray(img).save(f"dataset/real/{i}.jpg")  # Ye 2 folders banata hai:

In [3]:
for i in range(50):
    img = np.random.randint(0, 100, (128,128,3), dtype=np.uint8)
    Image.fromarray(img).save(f"dataset/fake/{i}.jpg")

In [4]:

import os

print("Real:", len(os.listdir("dataset/real")))
print("Fake:", len(os.listdir("dataset/fake")))    #Ye folder ke andar kitni images hain wo count karta hai.

Real: 50
Fake: 50


In [5]:
import cv2         
import numpy as np

X = []
y = []

for img in os.listdir("dataset/real"):
    img_path = "dataset/real/" + img
    image = cv2.imread(img_path)        #OpenCV image read karta hai.
    X.append(image)
    y.append(1)

for img in os.listdir("dataset/fake"):
    img_path = "dataset/fake/" + img
    image = cv2.imread(img_path)
    X.append(image)
    y.append(0)

X = np.array(X)/255.0
y = np.array(y)

print(X.shape, y.shape)

(100, 128, 128, 3) (100,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(80, 128, 128, 3) (20, 128, 128, 3)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)), #Ye feature detect karta hai. like edges,eyes,nose,texture
    MaxPooling2D(2,2),       #Image size chhoti karta hai.

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=8
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 235ms/step - accuracy: 0.5625 - loss: 0.6493 - val_accuracy: 1.0000 - val_loss: 0.5940
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 200ms/step - accuracy: 0.8875 - loss: 0.2781 - val_accuracy: 1.0000 - val_loss: 0.0124
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 190ms/step - accuracy: 1.0000 - loss: 0.0032 - val_accuracy: 1.0000 - val_loss: 2.5996e-06
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 194ms/step - accuracy: 1.0000 - loss: 6.4681e-05 - val_accuracy: 1.0000 - val_loss: 7.8758e-07
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 188ms/step - accuracy: 1.0000 - loss: 4.1479e-06 - val_accuracy: 1.0000 - val_loss: 2.0281e-07
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 191ms/step - accuracy: 1.0000 - loss: 1.0586e-05 - val_accuracy: 1.0000 - val_loss: 6.1992e-09
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 188ms/step - accuracy: 1.0000 - loss: 1.6053e-07 - val_accuracy: 1.0000 - val_loss: 4.9379e-10
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 189ms/step - accuracy: 1.00

In [11]:
y_pred = model.predict(X_test[:10])
print(y_pred)
print(y_test[:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
[[1.0000000e+00]
 [1.0000000e+00]
 [1.0000000e+00]
 [1.0000000e+00]
 [1.2564846e-10]
 [1.1542210e-10]
 [1.2004793e-10]
 [1.0000000e+00]
 [3.1759842e-10]
 [1.0966364e-10]]
[1 1 1 1 0 0 0 1 0 0]


In [12]:
pred = model.predict(X_test)

for i in pred:
    if i > 0.5:
        print("REAL")
    else:
        print("FAKE")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step
REAL
REAL
REAL
REAL
FAKE
FAKE
FAKE
REAL
FAKE
FAKE
FAKE
REAL
FAKE
FAKE
FAKE
REAL
REAL
FAKE
REAL
REAL


In [13]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [14]:
from sklearn.metrics import classification_report

print(classification_report(y_test, (pred > 0.5)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [15]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 1.0000 - loss: 8.0774e-11
Accuracy: 1.0


In [16]:
import cv2
import numpy as np

def predict_image(model, img_path):

    img = cv2.imread(img_path)
    
    if img is None:
        print("Image load nahi hui ❌")
        return

    img = cv2.resize(img, (128,128))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img)

    print("Prediction value:", pred[0][0])

    if pred[0][0] > 0.5:
        print("👉 REAL")
    else:
        print("👉 FAKE")

In [17]:
predict_image(model, r"C:\Users\Lenovo\OneDrive\Pictures\Camera Roll 1\Screenshot 2026-05-30 144422.png")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Prediction value: 1.0
👉 REAL


In [18]:
import numpy as np
import cv2

# fake-like noisy image (model test only)
img = np.random.randint(0, 80, (128,128,3), dtype=np.uint8)

cv2.imwrite("fake_test.jpg", img)

predict_image(model, "fake_test.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
Prediction value: 4.4298507e-10
👉 FAKE


In [24]:
import tensorflow as tf
model = tf.keras.models.load_model(
    r"C:\Users\Lenovo\deepfake_model.h5"
)
model.save(r"C:\Users\Lenovo\deepfake_model.h5")

In [ ]:
import streamlit as st
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image

# Page Title
st.set_page_config(page_title="Deepfake Detection", page_icon="🤖")

st.title("🤖 Deepfake Detection Using CNN")
st.write("Upload an image and check whether it is REAL or FAKE.")

# Load Model
model = tf.keras.models.load_model(r"C:\Users\Lenovo\deepfake_model.h5")

# Upload Image
uploaded_file = st.file_uploader(
    "Choose an Image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    image = Image.open(uploaded_file)

    st.image(image, caption="Uploaded Image", use_container_width=True)

    img = np.array(image)

    # Convert RGB to BGR (OpenCV format)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    img = cv2.resize(img, (128, 128))

    img = img / 255.0

    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)[0][0]

    st.write(f"Prediction Score : **{prediction:.6f}**")

    if prediction > 0.5:
        st.success("✅ REAL IMAGE")
        st.write(f"Confidence : **{prediction*100:.2f}%**")
    else:
        st.error("❌ FAKE IMAGE")
        st.write(f"Confidence : **{(1-prediction)*100:.2f}%**")
